In [1]:
import pandas as pd
import numpy as np

In [7]:
# =========================================================
# 1. 설정값
# =========================================================

CSV_PATH = "/content/sample_data/MOTOR_DATA.csv"

N_MOTORS = 4 # 모터 개수
BATTERY_CAPACITY_AH = 5.0 # 배터리 용량
USABLE_RATIO = 0.8 # 사용 가능한 배터리 용량을 80%로 가정
USABLE_CAPACITY_AH = BATTERY_CAPACITY_AH * USABLE_RATIO

# 미션 조건, 가중치 정의
MISSION_PAYLOAD = {
    "name": "payload",
    "target_thrust_g": 1000,      # 4kg / 4 motors
    "target_TR": 2.2,
    "mission_weight": 0.4,
    "weights": {
        "TR": 0.45,
        "throttle": 0.35,
        "power": 0.20
    }
}

MISSION_HOVERING = {
    "name": "hovering",
    "target_thrust_g": 625,       # 2.5kg / 4 motors
    "target_TR": 2.2,
    "mission_weight": 0.6,
    "weights": {
        "hover_time": 0.50,
        "eff_range": 0.20,
        "throttle": 0.15,
        "TR": 0.15
    }
}

# 비대칭 추력비 패널티 계수
# TR < 2.2이면 크게 감점, TR >= 2.2이면 완만하게 감점
K_LOW = 80
K_HIGH = 50

# 스로틀 기준 초과 패널티 기준
PAYLOAD_THROTTLE_LIMIT = 75   # 1000g 호버 스로틀이 75% 이하이면 100점
HOVER_THROTTLE_LIMIT = 60     # 625g 호버 스로틀이 60% 이하이면 100점
THROTTLE_MAX = 100            # 100%에서 0점

ALLOW_EXTRAPOLATE = True # 보간 허용

In [8]:
# =========================================================
# 2. 기본 함수
# =========================================================

def interp_at_thrust(thrust_array, value_array, target_thrust, allow_extrapolate=True):
    """
    Thrust(G)를 기준으로 target_thrust에서의 값을 선형 보간.
    예: 625g 또는 1000g에서의 throttle, current, power 계산.
    """
    x = np.asarray(thrust_array, dtype=float)
    y = np.asarray(value_array, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return np.nan

    idx = np.argsort(x)
    x = x[idx]
    y = y[idx]

    if x[0] <= target_thrust <= x[-1]:
        return float(np.interp(target_thrust, x, y))

    if not allow_extrapolate:
        return np.nan

    # 아래쪽 외삽
    if target_thrust < x[0]:
        x1, x2 = x[0], x[1]
        y1, y2 = y[0], y[1]

    # 위쪽 외삽
    else:
        x1, x2 = x[-2], x[-1]
        y1, y2 = y[-2], y[-1]

    return float(y1 + (y2 - y1) * (target_thrust - x1) / (x2 - x1))


def minmax_score(series, higher_better=True):
    """
    Min-Max 정규화 후 0~100점 변환하는 함수
    higher_better=True  : 클수록 좋은 항목(정방향 정규화)
    higher_better=False : 작을수록 좋은 항목(역방향 정규화)
    """
    s = series.astype(float)

    mn = s.min(skipna=True)
    mx = s.max(skipna=True)

    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.ones(len(s)) * 50, index=s.index)

    score = (s - mn) / (mx - mn) * 100

    if higher_better:
        return score
    else:
        return 100 - score


def thrust_ratio_score(TR, target_TR=2.2, k_low=80, k_high=50):
    """
    비대칭 추력비 점수를 계산하는 함수
    TR < 2.2이면 크게 감점,
    TR >= 2.2이면 감점은 하되 더 완만하게 감점.
    """
    if pd.isna(TR):
        return np.nan

    if TR < target_TR:
        score = 100 - k_low * (target_TR - TR)
    else:
        score = 100 - k_high * (TR - target_TR)

    return max(0, min(100, score))


def throttle_penalty_score(throttle, limit, throttle_max=100):
    """
    기준 초과 패널티.
    throttle <= limit이면 100점.
    limit보다 커질수록 선형 감점.
    throttle_max에서 0점.
    """
    if pd.isna(throttle):
        return np.nan

    if throttle <= limit:
        return 100

    score = 100 * (throttle_max - throttle) / (throttle_max - limit)
    return max(0, min(100, score))


def high_eff_range_score(g, target_thrust):
    """
    고효율 구간 점수.
    기준: 해당 모터-프롭 조합의 최대 효율의 90% 이상인 구간.

    target_thrust가 고효율 구간 안에 있으면 100점.
    밖에 있으면 가장 가까운 고효율 추력점과의 거리 기반 감점.
    """
    max_eff = g["Efficiency(G/W)"].max()
    threshold = 0.9 * max_eff

    high_eff_data = g[g["Efficiency(G/W)"] >= threshold]

    if high_eff_data.empty:
        return np.nan, np.nan, np.nan, False

    high_min = high_eff_data["Thrust(G)"].min()
    high_max = high_eff_data["Thrust(G)"].max()

    in_high_eff_range = high_min <= target_thrust <= high_max

    if in_high_eff_range:
        score = 100
        nearest_high_eff_thrust = target_thrust
    else:
        # 고효율 구간과 목표 추력 사이의 가장 가까운 거리
        if target_thrust < high_min:
            nearest_high_eff_thrust = high_min
        else:
            nearest_high_eff_thrust = high_max

        distance = abs(nearest_high_eff_thrust - target_thrust)

        # PDF의 거리 기반 감점 개념 반영
        score = max(0, 100 - (distance / target_thrust) * 100)

    return score, high_min, high_max, in_high_eff_range

In [9]:
# =========================================================
# 3. 데이터 불러오기
# =========================================================

df = pd.read_csv(CSV_PATH)

print("CSV columns:")
print(df.columns.tolist())

CSV columns:
['Motor', 'KV', 'Prop', 'Throttle(%)', 'Current(A)', 'Power(W)', 'Thrust(G)', 'RPM', 'Efficiency(G/W)']


In [10]:
# =========================================================
# 4. 모터-프롭 조합별 기본 성능 계산
# =========================================================

rows = []

for (motor, kv, prop), g in df.groupby(["Motor", "KV", "Prop"]):
    g = g.sort_values("Thrust(G)").copy()

    thrust = g["Thrust(G)"].values
    max_thrust = g["Thrust(G)"].max()

    row = {
        "Motor": motor,
        "KV": kv,
        "Prop": prop,
        "max_thrust_g_per_motor": max_thrust
    }

    # -------------------------
    # Mission 1: Payload, 1000g
    # -------------------------
    target_1000 = MISSION_PAYLOAD["target_thrust_g"]

    row["payload_possible"] = max_thrust >= target_1000

    row["payload_throttle_%"] = interp_at_thrust(
        thrust, g["Throttle(%)"], target_1000, ALLOW_EXTRAPOLATE
    )

    row["payload_current_A_per_motor"] = interp_at_thrust(
        thrust, g["Current(A)"], target_1000, ALLOW_EXTRAPOLATE
    )

    row["payload_power_W_per_motor"] = interp_at_thrust(
        thrust, g["Power(W)"], target_1000, ALLOW_EXTRAPOLATE
    )

    row["payload_efficiency_g_per_W"] = (
        target_1000 / row["payload_power_W_per_motor"]
        if pd.notna(row["payload_power_W_per_motor"]) and row["payload_power_W_per_motor"] > 0
        else np.nan
    )

    row["payload_TR_4kg"] = max_thrust / target_1000

    # -------------------------
    # Mission 2: Hovering, 625g
    # -------------------------
    target_625 = MISSION_HOVERING["target_thrust_g"]

    row["hovering_possible"] = max_thrust >= target_625

    row["hovering_throttle_%"] = interp_at_thrust(
        thrust, g["Throttle(%)"], target_625, ALLOW_EXTRAPOLATE
    )

    row["hovering_current_A_per_motor"] = interp_at_thrust(
        thrust, g["Current(A)"], target_625, ALLOW_EXTRAPOLATE
    )

    row["hovering_power_W_per_motor"] = interp_at_thrust(
        thrust, g["Power(W)"], target_625, ALLOW_EXTRAPOLATE
    )

    row["hovering_efficiency_g_per_W"] = (
        target_625 / row["hovering_power_W_per_motor"]
        if pd.notna(row["hovering_power_W_per_motor"]) and row["hovering_power_W_per_motor"] > 0
        else np.nan
    )

    row["hovering_total_current_A"] = (
        row["hovering_current_A_per_motor"] * N_MOTORS
        if pd.notna(row["hovering_current_A_per_motor"])
        else np.nan
    )

    row["hovering_time_min"] = (
        USABLE_CAPACITY_AH / row["hovering_total_current_A"] * 60
        if pd.notna(row["hovering_total_current_A"]) and row["hovering_total_current_A"] > 0
        else np.nan
    )

    row["hovering_TR_2p5kg"] = max_thrust / target_625

    eff_score, high_min, high_max, in_range = high_eff_range_score(g, target_625)

    row["hovering_high_eff_range_score"] = eff_score
    row["hovering_high_eff_range_min_g"] = high_min
    row["hovering_high_eff_range_max_g"] = high_max
    row["hovering_in_high_eff_range"] = in_range

    rows.append(row)

result = pd.DataFrame(rows)


In [11]:
# =========================================================
# 5. 전제 조건 불만족 후보 처리
# =========================================================

# PDF 기준:
# Mission 1: Max Thrust >= 1000g 불만족 시 제외
# Mission 2: Max Thrust >= 625g 불만족 시 제외
result["valid_for_payload"] = result["payload_possible"]
result["valid_for_hovering"] = result["hovering_possible"]


In [12]:
# =========================================================
# 6. Mission 1 점수 계산
# =========================================================

# 1) 4kg 기준 추력비 점수
result["payload_score_TR"] = result["payload_TR_4kg"].apply(
    lambda x: thrust_ratio_score(
        x,
        target_TR=MISSION_PAYLOAD["target_TR"],
        k_low=K_LOW,
        k_high=K_HIGH
    )
)

# 2) 1000g 호버 스로틀 점수: 기준 초과 패널티
result["payload_score_throttle"] = result["payload_throttle_%"].apply(
    lambda x: throttle_penalty_score(
        x,
        limit=PAYLOAD_THROTTLE_LIMIT,
        throttle_max=THROTTLE_MAX
    )
)

# 3) 1000g 호버 소비전력 점수: 낮을수록 좋음 → Min-Max 역방향 정규화
result["payload_score_power"] = minmax_score(
    result["payload_power_W_per_motor"],
    higher_better=False
)

# Payload mission 총점
w_payload = MISSION_PAYLOAD["weights"]

result["score_payload"] = (
    w_payload["TR"] * result["payload_score_TR"]
    + w_payload["throttle"] * result["payload_score_throttle"]
    + w_payload["power"] * result["payload_score_power"]
)

# 전제 조건 불만족 시 제외 처리
result.loc[~result["valid_for_payload"], "score_payload"] = np.nan


# =========================================================
# 7. Mission 2 점수 계산
# =========================================================

# 1) 625g 예상 호버링 시간: 클수록 좋음 → Min-Max 정규화
result["hovering_score_time"] = minmax_score(
    result["hovering_time_min"],
    higher_better=True
)

# 2) 625g 고효율 구간 포함 여부: 거리 기반 감점
result["hovering_score_eff_range"] = result["hovering_high_eff_range_score"]

# 3) 625g 호버 스로틀: 기준 초과 패널티
result["hovering_score_throttle"] = result["hovering_throttle_%"].apply(
    lambda x: throttle_penalty_score(
        x,
        limit=HOVER_THROTTLE_LIMIT,
        throttle_max=THROTTLE_MAX
    )
)

# 4) 2.5kg 기준 추력비 점수
result["hovering_score_TR"] = result["hovering_TR_2p5kg"].apply(
    lambda x: thrust_ratio_score(
        x,
        target_TR=MISSION_HOVERING["target_TR"],
        k_low=K_LOW,
        k_high=K_HIGH
    )
)

# Hovering mission 총점
w_hover = MISSION_HOVERING["weights"]

result["score_hovering"] = (
    w_hover["hover_time"] * result["hovering_score_time"]
    + w_hover["eff_range"] * result["hovering_score_eff_range"]
    + w_hover["throttle"] * result["hovering_score_throttle"]
    + w_hover["TR"] * result["hovering_score_TR"]
)

# 전제 조건 불만족 시 제외 처리
result.loc[~result["valid_for_hovering"], "score_hovering"] = np.nan


In [13]:
# =========================================================
# 8. 최종 점수 계산
# =========================================================

result["score_total"] = (
    MISSION_PAYLOAD["mission_weight"] * result["score_payload"]
    + MISSION_HOVERING["mission_weight"] * result["score_hovering"]
)

result = result.sort_values("score_total", ascending=False)

In [16]:
# =========================================================
# 9. 결과 출력 및 저장
# =========================================================

# 반올림한 복사본 생성
result_view = result.copy()

# ---------------------------------------------------------
# 9-1. 제외 사유 정리
# ---------------------------------------------------------

def exclusion_reason(row):
    reasons = []

    if not row["valid_for_payload"]:
        reasons.append("Mission 1 제외: 최대추력 < 1000g")

    if not row["valid_for_hovering"]:
        reasons.append("Mission 2 제외: 최대추력 < 625g")

    if len(reasons) == 0:
        return "통과"
    else:
        return " / ".join(reasons)

result_view["exclusion_reason"] = result_view.apply(exclusion_reason, axis=1)


# ---------------------------------------------------------
# 9-2. Mission 1 Payload 점수표
# ---------------------------------------------------------

mission1_cols = [
    "Motor",
    "KV",
    "Prop",
    "max_thrust_g_per_motor",
    "payload_TR_4kg",
    "payload_throttle_%",
    "payload_power_W_per_motor",
    "payload_score_TR",
    "payload_score_throttle",
    "payload_score_power",
    "score_payload",
    "valid_for_payload"
]

mission1_result = result_view[mission1_cols].copy()
mission1_result = mission1_result.sort_values("score_payload", ascending=False)

print("\n" + "=" * 80)
print("Mission 1: Payload Mission Score")
print("=" * 80)
print("평가 조건: 1000g hover 가능 여부, 4kg 기준 추력비, 1000g 스로틀, 1000g 소비전력")
print("전제 조건: Max Thrust >= 1000g")
print("-" * 80)

print(mission1_result.round(3).to_string(index=False))


# ---------------------------------------------------------
# 9-3. Mission 2 Hovering 점수표
# ---------------------------------------------------------

mission2_cols = [
    "Motor",
    "KV",
    "Prop",
    "max_thrust_g_per_motor",
    "hovering_TR_2p5kg",
    "hovering_throttle_%",
    "hovering_power_W_per_motor",
    "hovering_time_min",
    "hovering_in_high_eff_range",
    "hovering_high_eff_range_min_g",
    "hovering_high_eff_range_max_g",
    "hovering_score_time",
    "hovering_score_eff_range",
    "hovering_score_throttle",
    "hovering_score_TR",
    "score_hovering",
    "valid_for_hovering"
]

mission2_result = result_view[mission2_cols].copy()
mission2_result = mission2_result.sort_values("score_hovering", ascending=False)

print("\n" + "=" * 80)
print("Mission 2: Hovering Mission Score")
print("=" * 80)
print("평가 조건: 625g 예상 호버링 시간, 고효율 구간 포함 여부, 625g 스로틀, 2.5kg 기준 추력비")
print("전제 조건: Max Thrust >= 625g")
print("-" * 80)

print(mission2_result.round(3).to_string(index=False))


# ---------------------------------------------------------
# 9-4. 제외된 후보 및 제외 이유
# ---------------------------------------------------------

excluded_result = result_view[result_view["exclusion_reason"] != "통과"].copy()

excluded_cols = [
    "Motor",
    "KV",
    "Prop",
    "max_thrust_g_per_motor",
    "payload_possible",
    "hovering_possible",
    "exclusion_reason"
]

print("\n" + "=" * 80)
print("Excluded Candidates")
print("=" * 80)

if excluded_result.empty:
    print("제외된 후보 없음")
else:
    print(excluded_result[excluded_cols].to_string(index=False))


# ---------------------------------------------------------
# 9-5. 최종 종합 순위표
# ---------------------------------------------------------

final_cols = [
    "Motor",
    "KV",
    "Prop",
    "max_thrust_g_per_motor",
    "score_payload",
    "score_hovering",
    "score_total",
    "exclusion_reason"
]

final_result = result_view[final_cols].copy()
final_result = final_result.sort_values("score_total", ascending=False)

print("\n" + "=" * 80)
print("Final Motor Selection Result")
print("=" * 80)
print("최종 점수 = 0.4 × Mission 1 Payload Score + 0.6 × Mission 2 Hovering Score")
print("-" * 80)

print(final_result.round(3).to_string(index=False))


# ---------------------------------------------------------
# 9-6. 1등 후보 따로 출력
# ---------------------------------------------------------

best = final_result.iloc[0]

print("\n" + "=" * 80)
print("Best Candidate")
print("=" * 80)
print(f"최종 선정 후보: {best['Motor']} / KV {best['KV']} / Prop {best['Prop']}")
print(f"Mission 1 Score: {best['score_payload']:.3f}")
print(f"Mission 2 Score: {best['score_hovering']:.3f}")
print(f"Final Score: {best['score_total']:.3f}")
print(f"판정: {best['exclusion_reason']}")


# ---------------------------------------------------------
# 9-7. CSV 파일로 각각 저장
# ---------------------------------------------------------

mission1_result.to_csv(
    "mission1_payload_score.csv",
    index=False,
    encoding="utf-8-sig"
)

mission2_result.to_csv(
    "mission2_hovering_score.csv",
    index=False,
    encoding="utf-8-sig"
)

excluded_result.to_csv(
    "excluded_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

final_result.to_csv(
    "final_motor_selection_result.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved files:")
print("- mission1_payload_score.csv")
print("- mission2_hovering_score.csv")
print("- excluded_candidates.csv")
print("- final_motor_selection_result.csv")


Mission 1: Payload Mission Score
평가 조건: 1000g hover 가능 여부, 4kg 기준 추력비, 1000g 스로틀, 1000g 소비전력
전제 조건: Max Thrust >= 1000g
--------------------------------------------------------------------------------
 Motor  KV   Prop  max_thrust_g_per_motor  payload_TR_4kg  payload_throttle_%  payload_power_W_per_motor  payload_score_TR  payload_score_throttle  payload_score_power  score_payload  valid_for_payload
MN3510 700 13*4.4                    1800            1.80              60.385                    113.618              68.0                 100.000              100.000         85.600               True
MN4010 580 13*4.4                    1420            1.42              73.571                    118.400              37.6                 100.000               86.093         69.139               True
MN3510 700   12*4                    1600            1.60              70.455                    132.931              52.0                 100.000               43.829         67.166          